In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
df= pd.read_csv("data/processed_rat_data.csv")


3. Feed Intake Volatility: Stable vs Erratic Intake
What we're measuring:

Within-rat variability: For each individual rat, calculate the standard deviation (SD) or coefficient of variation (CV) of their weekly feed intake across 15 weeks
CV = (SD / mean) × 100 ← better than SD because it's normalized (a rat eating 20g with SD=2 is more stable than a rat eating 10g with SD=2)

Classification:

Stable: Low CV (e.g., CV < 15%)
Erratic: High CV (e.g., CV > 25%)
Moderate: In between

Key questions:

Which water groups produce more stable feeders?
Are stable feeders also the ones with better final weight outcomes?
Does gender affect volatility? (females might be more consistent?)
Do fasting groups inherently have higher volatility, or do they stabilize between fasting cycles?
Is early volatility (weeks 1-5) predictive of final outcomes?

In [2]:
# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Create output directory
output_dir = Path('outputs')

print(f"Data loaded: {len(df)} rows")
print(f"Unique rats: {df['unique_rat_id'].nunique()}")

# ============================================================================
# STEP 1: Calculate Volatility Metrics for Each Rat
# ============================================================================
print("\n" + "="*80)
print("STEP 1: Calculating Feed Intake Volatility Metrics")
print("="*80)

# Create group_short if needed
def shorten_group_name(name):
    if 'Group 1' in name and 'RO water with < 20' in name:
        return 'G01: RO <20 TDS'
    elif 'Group 2' in name:
        return 'G02: RO 50-75 TDS'
    elif 'Group 3' in name and 'alternate' not in name.lower():
        return 'G03: RO 125-150 TDS'
    elif 'Group 4' in name:
        return 'G04: Telugu Ganga'
    elif 'Group 5' in name and 'alternate' not in name.lower():
        return 'G05: Kalyani Dam'
    elif 'Group 6' in name and 'alternate' not in name.lower():
        return 'G06: Ground Water'
    elif 'Group 7' in name:
        return 'G07: RO <20 (Fasting)'
    elif 'Group 8' in name:
        return 'G08: Kalyani (Fasting)'
    elif 'Group 9' in name:
        return 'G09: BIS Standard'
    elif 'Group 10' in name:
        return 'G10: Ground (Fasting)'
    elif 'Group 11' in name:
        return 'G11: RO 125-150 (Fasting)'
    return name

if 'group_short' not in df.columns:
    df['group_short'] = df['water_group'].apply(shorten_group_name)

# Calculate volatility metrics per rat
print("\nCalculating volatility metrics per rat...")

volatility_data = []

for rat_id in df['unique_rat_id'].unique():
    rat_data = df[df['unique_rat_id'] == rat_id].sort_values('week')
    
    # Overall metrics (all 15 weeks)
    feed_intake = rat_data['weekly_feed_intake'].values
    mean_intake = np.mean(feed_intake)
    sd_intake = np.std(feed_intake, ddof=1)  # Sample standard deviation
    cv_intake = (sd_intake / mean_intake) * 100  # Coefficient of variation
    
    # Early phase metrics (weeks 1-5)
    early_data = rat_data[rat_data['week'] <= 5]['weekly_feed_intake'].values
    early_mean = np.mean(early_data)
    early_sd = np.std(early_data, ddof=1)
    early_cv = (early_sd / early_mean) * 100
    
    # Get metadata
    gender = rat_data['gender'].iloc[0]
    group = rat_data['group_short'].iloc[0]
    initial_weight = rat_data['initial_body_weight'].iloc[0]
    final_weight = rat_data['final_body_weight'].iloc[0] if 'final_body_weight' in rat_data.columns else None
    total_weight_gain = final_weight - initial_weight if final_weight else None
    
    # Classify stability
    if cv_intake < 15:
        stability_class = 'Stable'
    elif cv_intake > 25:
        stability_class = 'Erratic'
    else:
        stability_class = 'Moderate'
    
    volatility_data.append({
        'unique_rat_id': rat_id,
        'gender': gender,
        'group_short': group,
        'mean_intake': mean_intake,
        'sd_intake': sd_intake,
        'cv_intake': cv_intake,
        'early_mean': early_mean,
        'early_sd': early_sd,
        'early_cv': early_cv,
        'stability_class': stability_class,
        'initial_weight': initial_weight,
        'final_weight': final_weight,
        'total_weight_gain': total_weight_gain
    })

volatility_df = pd.DataFrame(volatility_data)

# Sort groups
volatility_df['group_short'] = pd.Categorical(
    volatility_df['group_short'],
    categories=sorted(volatility_df['group_short'].unique()),
    ordered=True
)

print(f"\nRats analyzed: {len(volatility_df)}")
print(f"\nStability distribution:")
print(volatility_df['stability_class'].value_counts())

print(f"\nCV statistics:")
print(f"  Mean CV: {volatility_df['cv_intake'].mean():.2f}%")
print(f"  Median CV: {volatility_df['cv_intake'].median():.2f}%")
print(f"  Min CV: {volatility_df['cv_intake'].min():.2f}%")
print(f"  Max CV: {volatility_df['cv_intake'].max():.2f}%")

# Save to CSV
volatility_df.to_csv(output_dir / 'feed_intake_volatility_per_rat.csv', index=False)
print(f"\n✓ Saved: feed_intake_volatility_per_rat.csv")

# ============================================================================
# STEP 2: Question 1 - Which water groups produce more stable feeders?
# ============================================================================
print("\n" + "="*80)
print("STEP 2: Stability by Water Group")
print("="*80)

# Summary by group
group_volatility = volatility_df.groupby('group_short').agg({
    'cv_intake': ['mean', 'median', 'std', 'min', 'max'],
    'stability_class': lambda x: (x == 'Stable').sum()
}).reset_index()

group_volatility.columns = ['group_short', 'mean_cv', 'median_cv', 'std_cv', 'min_cv', 'max_cv', 'stable_count']
group_volatility['total_rats'] = volatility_df.groupby('group_short').size().values
group_volatility['stable_percent'] = (group_volatility['stable_count'] / group_volatility['total_rats']) * 100

print("\n--- Volatility by Water Group ---")
print(group_volatility.sort_values('median_cv').to_string(index=False))

# Save
group_volatility.to_csv(output_dir / 'volatility_by_group.csv', index=False)
print(f"\n✓ Saved: volatility_by_group.csv")

# Most stable groups
print("\n--- Most Stable Groups (Lowest Median CV) ---")
print(group_volatility.sort_values('median_cv').head(5)[['group_short', 'median_cv', 'stable_percent']].to_string(index=False))

print("\n--- Least Stable Groups (Highest Median CV) ---")
print(group_volatility.sort_values('median_cv', ascending=False).head(5)[['group_short', 'median_cv', 'stable_percent']].to_string(index=False))

# ============================================================================
# STEP 3: Question 2 - Stable feeders vs final weight outcomes
# ============================================================================
print("\n" + "="*80)
print("STEP 3: Stability vs Final Weight Outcomes")
print("="*80)

# Correlation analysis
valid_data = volatility_df[volatility_df['total_weight_gain'].notna()]

corr_cv_weight = valid_data['cv_intake'].corr(valid_data['total_weight_gain'])
corr_cv_final = valid_data['cv_intake'].corr(valid_data['final_weight'])

print(f"\nCorrelation: CV vs Total Weight Gain: {corr_cv_weight:.3f}")
print(f"Correlation: CV vs Final Weight: {corr_cv_final:.3f}")

if abs(corr_cv_weight) < 0.3:
    print("  → Weak correlation - stability doesn't strongly predict weight gain")
elif abs(corr_cv_weight) < 0.7:
    print("  → Moderate correlation")
else:
    print("  → Strong correlation")

# Compare by stability class
print("\n--- Weight Outcomes by Stability Class ---")
stability_outcomes = valid_data.groupby('stability_class').agg({
    'total_weight_gain': ['mean', 'std', 'count'],
    'final_weight': ['mean', 'std']
}).round(2)
print(stability_outcomes)

# Statistical test
stable_weights = valid_data[valid_data['stability_class'] == 'Stable']['total_weight_gain']
erratic_weights = valid_data[valid_data['stability_class'] == 'Erratic']['total_weight_gain']

if len(stable_weights) > 0 and len(erratic_weights) > 0:
    t_stat, p_val = stats.ttest_ind(stable_weights, erratic_weights)
    print(f"\nT-test (Stable vs Erratic): t={t_stat:.3f}, p={p_val:.4f}")
    if p_val < 0.05:
        print("  → Significant difference in weight gain")
    else:
        print("  → No significant difference")

# ============================================================================
# STEP 4: Question 3 - Gender effects on volatility
# ============================================================================
print("\n" + "="*80)
print("STEP 4: Gender Effects on Volatility")
print("="*80)

# Compare by gender
print("\n--- Volatility by Gender ---")
gender_volatility = volatility_df.groupby('gender').agg({
    'cv_intake': ['mean', 'median', 'std'],
    'stability_class': lambda x: (x == 'Stable').sum()
}).reset_index()

gender_volatility.columns = ['gender', 'mean_cv', 'median_cv', 'std_cv', 'stable_count']
gender_volatility['total'] = volatility_df.groupby('gender').size().values
gender_volatility['stable_percent'] = (gender_volatility['stable_count'] / gender_volatility['total']) * 100

print(gender_volatility.to_string(index=False))

# Statistical test
male_cv = volatility_df[volatility_df['gender'] == 'male']['cv_intake']
female_cv = volatility_df[volatility_df['gender'] == 'female']['cv_intake']

t_stat, p_val = stats.ttest_ind(male_cv, female_cv)
print(f"\nT-test (Male vs Female CV): t={t_stat:.3f}, p={p_val:.4f}")
if p_val < 0.05:
    print("  → Significant gender difference in volatility")
else:
    print("  → No significant gender difference")

# Gender x Group interaction
print("\n--- Volatility by Gender and Group ---")
gender_group = volatility_df.groupby(['group_short', 'gender'])['cv_intake'].agg(['mean', 'median', 'count']).reset_index()
gender_group_pivot = gender_group.pivot(index='group_short', columns='gender', values='median')
print(gender_group_pivot.to_string())

gender_group.to_csv(output_dir / 'volatility_by_gender_group.csv', index=False)

# ============================================================================
# STEP 5: Question 4 - Fasting groups volatility
# ============================================================================
print("\n" + "="*80)
print("STEP 5: Fasting vs Non-Fasting Groups")
print("="*80)

# Identify fasting groups
volatility_df['is_fasting'] = volatility_df['group_short'].str.contains('Fasting')

print("\n--- Fasting vs Non-Fasting Volatility ---")
fasting_comparison = volatility_df.groupby('is_fasting').agg({
    'cv_intake': ['mean', 'median', 'std'],
    'stability_class': lambda x: (x == 'Stable').sum()
}).reset_index()

fasting_comparison.columns = ['is_fasting', 'mean_cv', 'median_cv', 'std_cv', 'stable_count']
fasting_comparison['total'] = volatility_df.groupby('is_fasting').size().values
fasting_comparison['stable_percent'] = (fasting_comparison['stable_count'] / fasting_comparison['total']) * 100
fasting_comparison['is_fasting'] = fasting_comparison['is_fasting'].map({True: 'Fasting', False: 'Non-Fasting'})

print(fasting_comparison.to_string(index=False))

# Statistical test
fasting_cv = volatility_df[volatility_df['is_fasting']]['cv_intake']
non_fasting_cv = volatility_df[~volatility_df['is_fasting']]['cv_intake']

t_stat, p_val = stats.ttest_ind(fasting_cv, non_fasting_cv)
print(f"\nT-test (Fasting vs Non-Fasting): t={t_stat:.3f}, p={p_val:.4f}")
if p_val < 0.05:
    print("  → Fasting significantly affects volatility")
else:
    print("  → No significant difference")

# ============================================================================
# STEP 6: Question 5 - Early volatility predicts outcomes?
# ============================================================================
print("\n" + "="*80)
print("STEP 6: Early Volatility (Weeks 1-5) as Predictor")
print("="*80)

# Correlation between early CV and final outcomes
valid_data = volatility_df[volatility_df['total_weight_gain'].notna()]

corr_early_final_cv = valid_data['early_cv'].corr(valid_data['cv_intake'])
corr_early_weight = valid_data['early_cv'].corr(valid_data['total_weight_gain'])

print(f"\nCorrelation: Early CV vs Overall CV: {corr_early_final_cv:.3f}")
print(f"Correlation: Early CV vs Total Weight Gain: {corr_early_weight:.3f}")

if abs(corr_early_weight) > 0.5:
    print("  → Early volatility is a good predictor of final weight gain")
elif abs(corr_early_weight) > 0.3:
    print("  → Early volatility has moderate predictive value")
else:
    print("  → Early volatility is a weak predictor")

# Classify by early CV
volatility_df['early_stable'] = volatility_df['early_cv'] < 15
valid_data = volatility_df[volatility_df['total_weight_gain'].notna()]

print("\n--- Final Outcomes by Early Stability ---")
early_outcomes = valid_data.groupby('early_stable').agg({
    'total_weight_gain': ['mean', 'std', 'count'],
    'cv_intake': ['mean', 'median']
}).round(2)
early_outcomes.index = ['Early Erratic (CV≥15%)', 'Early Stable (CV<15%)']
print(early_outcomes)

# ============================================================================
# STEP 7: Create Visualizations
# ============================================================================
print("\n" + "="*80)
print("STEP 7: Creating Visualizations")
print("="*80)

fig = plt.figure(figsize=(20, 14))
gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)

# ============================================================================
# PANEL 1: CV Distribution by Water Group (Box Plot)
# ============================================================================
print("\nCreating Panel 1: CV by water group...")
ax1 = fig.add_subplot(gs[0, :])

groups_ordered = sorted(volatility_df['group_short'].unique())
plot_data = [volatility_df[volatility_df['group_short'] == g]['cv_intake'].values for g in groups_ordered]

bp = ax1.boxplot(plot_data, 
                 labels=groups_ordered,
                 patch_artist=True,
                 showfliers=True)

for patch in bp['boxes']:
    patch.set_facecolor('#7FB3D5')
    patch.set_alpha(0.7)

ax1.axhline(y=15, color='green', linestyle='--', alpha=0.5, linewidth=2, label='Stable threshold (CV<15%)')
ax1.axhline(y=25, color='red', linestyle='--', alpha=0.5, linewidth=2, label='Erratic threshold (CV>25%)')

ax1.set_ylabel('Coefficient of Variation (%)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Water Group', fontsize=12, fontweight='bold')
ax1.set_title('Feed Intake Volatility by Water Group', fontsize=14, fontweight='bold')
ax1.tick_params(axis='x', rotation=45, labelsize=10)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3, axis='y')

# ============================================================================
# PANEL 2: CV vs Final Weight Gain (Scatter)
# ============================================================================
print("Creating Panel 2: CV vs weight gain scatter...")
ax2 = fig.add_subplot(gs[1, 0])

valid_data = volatility_df[volatility_df['total_weight_gain'].notna()]

# Color by stability class
colors = {'Stable': 'green', 'Moderate': 'orange', 'Erratic': 'red'}
for stability, color in colors.items():
    subset = valid_data[valid_data['stability_class'] == stability]
    ax2.scatter(subset['cv_intake'], subset['total_weight_gain'],
               alpha=0.6, s=50, c=color, label=stability, edgecolors='black', linewidth=0.5)

# Add trend line
z = np.polyfit(valid_data['cv_intake'], valid_data['total_weight_gain'], 1)
p = np.poly1d(z)
ax2.plot(valid_data['cv_intake'], p(valid_data['cv_intake']), 
         "k--", alpha=0.5, linewidth=2, label=f'Trend (r={corr_cv_weight:.2f})')

ax2.set_xlabel('Coefficient of Variation (%)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Total Weight Gain (g)', fontsize=12, fontweight='bold')
ax2.set_title('Feed Intake Volatility vs Weight Gain', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

# ============================================================================
# PANEL 3: Gender Comparison (Violin Plot)
# ============================================================================
print("Creating Panel 3: Gender comparison...")
ax3 = fig.add_subplot(gs[1, 1])

parts = ax3.violinplot([male_cv, female_cv], 
                       positions=[1, 2],
                       showmeans=True,
                       showmedians=True)

for pc in parts['bodies']:
    pc.set_facecolor('#D5A6BD')
    pc.set_alpha(0.7)

ax3.set_xticks([1, 2])
ax3.set_xticklabels(['Male', 'Female'])
ax3.set_ylabel('Coefficient of Variation (%)', fontsize=12, fontweight='bold')
ax3.set_title('Feed Intake Volatility by Gender', fontsize=13, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='y')

# Add mean values as text
ax3.text(1, male_cv.max() + 2, f'Mean: {male_cv.mean():.1f}%', ha='center', fontsize=10)
ax3.text(2, female_cv.max() + 2, f'Mean: {female_cv.mean():.1f}%', ha='center', fontsize=10)

# ============================================================================
# PANEL 4: Fasting vs Non-Fasting (Bar Chart)
# ============================================================================
print("Creating Panel 4: Fasting comparison...")
ax4 = fig.add_subplot(gs[2, 0])

fasting_means = fasting_comparison.set_index('is_fasting')['mean_cv']
fasting_stds = volatility_df.groupby(volatility_df['group_short'].str.contains('Fasting'))['cv_intake'].std().values

x = np.arange(len(fasting_means))
bars = ax4.bar(x, fasting_means.values, yerr=fasting_stds, 
               color=['#FF6B6B', '#4ECDC4'], alpha=0.7, capsize=5)

ax4.set_xticks(x)
ax4.set_xticklabels(fasting_means.index)
ax4.set_ylabel('Mean CV (%)', fontsize=12, fontweight='bold')
ax4.set_title('Volatility: Fasting vs Non-Fasting Groups', fontsize=13, fontweight='bold')
ax4.grid(True, alpha=0.3, axis='y')

# Add values on bars
for i, (bar, val) in enumerate(zip(bars, fasting_means.values)):
    ax4.text(bar.get_x() + bar.get_width()/2, val + 1, f'{val:.1f}%',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

# ============================================================================
# PANEL 5: Early CV vs Overall CV (Scatter)
# ============================================================================
print("Creating Panel 5: Early vs overall CV...")
ax5 = fig.add_subplot(gs[2, 1])

ax5.scatter(volatility_df['early_cv'], volatility_df['cv_intake'],
           alpha=0.6, s=50, c='#9B59B6', edgecolors='black', linewidth=0.5)

# Add trend line
z = np.polyfit(volatility_df['early_cv'], volatility_df['cv_intake'], 1)
p = np.poly1d(z)
ax5.plot(volatility_df['early_cv'], p(volatility_df['early_cv']), 
         "k--", alpha=0.5, linewidth=2, label=f'Trend (r={corr_early_final_cv:.2f})')

# Add diagonal reference line
max_val = max(volatility_df['early_cv'].max(), volatility_df['cv_intake'].max())
ax5.plot([0, max_val], [0, max_val], 'r:', alpha=0.3, linewidth=1, label='Perfect prediction')

ax5.set_xlabel('Early CV (Weeks 1-5) %', fontsize=12, fontweight='bold')
ax5.set_ylabel('Overall CV (All Weeks) %', fontsize=12, fontweight='bold')
ax5.set_title('Early Volatility as Predictor of Overall Volatility', fontsize=13, fontweight='bold')
ax5.legend(fontsize=10)
ax5.grid(True, alpha=0.3)

# ============================================================================
# Save Figure
# ============================================================================
plt.suptitle('Feed Intake Volatility Analysis', fontsize=16, fontweight='bold', y=0.995)

output_path = output_dir / 'feed_intake_volatility_analysis.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"\n✓ Saved: {output_path}")
plt.close()

# ============================================================================
# STEP 8: Generate Summary Report
# ============================================================================
print("\n" + "="*80)
print("FINAL SUMMARY - FEED INTAKE VOLATILITY")
print("="*80)

print("\n" + "="*50)
print("QUESTION 1: Most Stable Water Groups")
print("="*50)
top3_stable = group_volatility.sort_values('median_cv').head(3)
for idx, row in top3_stable.iterrows():
    print(f"{row['group_short']}: Median CV = {row['median_cv']:.1f}%, {row['stable_percent']:.0f}% stable rats")

print("\n" + "="*50)
print("QUESTION 2: Stability vs Weight Outcomes")
print("="*50)
print(f"Correlation (CV vs Weight Gain): {corr_cv_weight:.3f}")
if abs(corr_cv_weight) < 0.3:
    print("→ Weak relationship - stability doesn't strongly predict weight")
else:
    print("→ Moderate to strong relationship detected")

print("\n" + "="*50)
print("QUESTION 3: Gender Effects")
print("="*50)
print(f"Male mean CV: {male_cv.mean():.2f}%")
print(f"Female mean CV: {female_cv.mean():.2f}%")
print(f"Statistical significance: p = {p_val:.4f}")

print("\n" + "="*50)
print("QUESTION 4: Fasting Impact")
print("="*50)
print(fasting_comparison[['is_fasting', 'mean_cv', 'stable_percent']].to_string(index=False))

print("\n" + "="*50)
print("QUESTION 5: Early Volatility Prediction")
print("="*50)
print(f"Correlation (Early CV vs Overall CV): {corr_early_final_cv:.3f}")
print(f"Correlation (Early CV vs Weight Gain): {corr_early_weight:.3f}")

print("\n" + "="*80)
print("ANALYSIS COMPLETE!")
print("="*80)
print("\nGenerated files:")
print("  1. feed_intake_volatility_analysis.png - Main visualization")
print("  2. feed_intake_volatility_per_rat.csv - Individual rat data")
print("  3. volatility_by_group.csv - Group-level summary")
print("  4. volatility_by_gender_group.csv - Gender × Group interaction")
print("\n" + "="*80)

Data loaded: 1650 rows
Unique rats: 110

STEP 1: Calculating Feed Intake Volatility Metrics

Calculating volatility metrics per rat...

Rats analyzed: 110

Stability distribution:
stability_class
Stable      73
Moderate    35
Erratic      2
Name: count, dtype: int64

CV statistics:
  Mean CV: 14.45%
  Median CV: 13.68%
  Min CV: 9.06%
  Max CV: 29.13%

✓ Saved: feed_intake_volatility_per_rat.csv

STEP 2: Stability by Water Group

--- Volatility by Water Group ---
              group_short   mean_cv  median_cv   std_cv    min_cv    max_cv  stable_count  total_rats  stable_percent
    G10: Ground (Fasting)  9.895868   9.616411 1.062283  9.058099 12.203931            10          10           100.0
G11: RO 125-150 (Fasting) 11.329384  10.791320 1.965604  9.058099 14.003862            10          10           100.0
   G08: Kalyani (Fasting) 11.694032  11.158271 1.685538 10.399629 16.076805             9          10            90.0
    G07: RO <20 (Fasting) 12.722777  13.030280 1.969552 10.3

/var/folders/h9/24vbd1m931s_2j_jt_vy8cyr0000gn/T/ipykernel_21969/4103607574.py:129: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_volatility = volatility_df.groupby('group_short').agg({
/var/folders/h9/24vbd1m931s_2j_jt_vy8cyr0000gn/T/ipykernel_21969/4103607574.py:135: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_volatility['total_rats'] = volatility_df.groupby('group_short').size().values
/var/folders/h9/24vbd1m931s_2j_jt_vy8cyr0000gn/T/ipykernel_21969/4103607574.py:228: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass obs


✓ Saved: outputs/feed_intake_volatility_analysis.png

FINAL SUMMARY - FEED INTAKE VOLATILITY

QUESTION 1: Most Stable Water Groups
G10: Ground (Fasting): Median CV = 9.6%, 100% stable rats
G11: RO 125-150 (Fasting): Median CV = 10.8%, 100% stable rats
G08: Kalyani (Fasting): Median CV = 11.2%, 90% stable rats

QUESTION 2: Stability vs Weight Outcomes
Correlation (CV vs Weight Gain): 0.469
→ Moderate to strong relationship detected

QUESTION 3: Gender Effects
Male mean CV: 14.45%
Female mean CV: 14.46%
Statistical significance: p = 0.0000

QUESTION 4: Fasting Impact
 is_fasting   mean_cv  stable_percent
Non-Fasting 16.191696       48.571429
    Fasting 11.410515       97.500000

QUESTION 5: Early Volatility Prediction
Correlation (Early CV vs Overall CV): 0.375
Correlation (Early CV vs Weight Gain): 0.466

ANALYSIS COMPLETE!

Generated files:
  1. feed_intake_volatility_analysis.png - Main visualization
  2. feed_intake_volatility_per_rat.csv - Individual rat data
  3. volatility_by_gr